#  Paradigma NoSQL y Modelado de Documentos

---

**Universidad de la Sabana**

**Asignatura:** Diseño y Optimización de Bases de Datos

**Actividad:** Guía de Actividad Unidad 3

**Fecha:** Semama del 15 al 22 de Junio  

---
### Integrantes del Equipo:
1. Juan Daniel Valderrama
2. Jorge Esteban Triviño Correa
3. Javier Andres Baron Fontanilla

---

In [ ]:
# Instalación, Configuración y Librerías

!pip install pymongo[srv] tqdm tabulate

import time
import pandas as pd
from pymongo import MongoClient
from urllib.parse import quote_plus
from google.colab import drive
from tqdm.notebook import tqdm
from tabulate import tabulate
from bson import json_util

# Configuración de conexión MongoDB - Corregido con quote_plus
user = "atlasAdmin"
password = "sUJ@23PO011"
host = "ecommify-u5.blcr7z7.mongodb.net"

# Codificación necesaria para evitar error en el símbolo '@'
MONGO_URI = f"mongodb+srv://{user}:{quote_plus(password)}@{host}/"
DB_NAME = "ecommify_db"
client = MongoClient(MONGO_URI)
db = client[DB_NAME]

def print_banner(titulo):
    print(f"\n" + "="*80)
    print(f"🚀 {titulo.upper()}")
    print("="*80)


# ETAPA 1: Infraestructura, Validación y Limpieza
def setup_collections():
    print_banner("Etapa 1: Preparación de Infraestructura y Esquemas")
    start_time = time.time()

    colecciones = ["products", "reviews"]
    existentes = db.list_collection_names()

    for col in colecciones:
        if col in existentes:
            db.drop_collection(col)
            print(f"⚠️  Colección '{col}' purgada para aplicar nuevo esquema.")

    # Esquema Products
    db.create_collection("products", validator={
        "$jsonSchema": {
            "bsonType": "object",
            "required": ["product_id_pg", "name", "display_price", "seller"],
            "properties": {
                "product_id_pg": {"bsonType": "string"},
                "name": {"bsonType": "string"},
                "display_price": {"bsonType": "number"},
                "seller": {
                    "bsonType": "object",
                    "required": ["seller_id_pg", "name", "reputation_score"],
                    "properties": {
                        "seller_id_pg": {"bsonType": "string"},
                        "name": {"bsonType": "string"},
                        "reputation_score": {"bsonType": "number"}
                    }
                }
            }
        }
    })

    # Esquema Reviews
    db.create_collection("reviews", validator={
        "$jsonSchema": {
            "bsonType": "object",
            "required": ["order_id_pg", "product_id_pg", "customer", "score", "creation_date"],
            "properties": {"score": {"bsonType": "int"}}
        }
    })
    print(f"✅ Esquemas JSON aplicados correctamente en {round(time.time() - start_time, 2)}s.")

# ETAPA 2: Procesamiento - Data Wrangling con Google Drive

def get_prepared_data():
    print_banner("Etapa 2: Extracción y Transformación de Datos")
    start_time = time.time()
    try:
        drive.mount('/content/drive', force_remount=True)
        base_path = '/content/drive/MyDrive/BD maestria/'

        print("📥 Cargando datasets desde Google Drive...")
        df_order_items = pd.read_csv(base_path + 'olist_order_items_dataset.csv')
        df_order_reviews = pd.read_csv(base_path + 'olist_order_reviews_dataset.csv')
        df_orders = pd.read_csv(base_path + 'olist_orders_dataset.csv')
        df_products = pd.read_csv(base_path + 'olist_products_dataset.csv')

        # --- Transformar Products ---
        print("⚙️  Procesando Catálogo de Productos...")
        lista_productos = []
        for _, row in tqdm(df_products.head(1000).iterrows(), total=1000, desc="Productos"):
            lista_productos.append({
                "product_id_pg": str(row['product_id']),
                "name": str(row['product_category_name']) if pd.notna(row['product_category_name']) else "Sin Categoria",
                "display_price": 50.0,
                "seller": {"seller_id_pg": "S_001", "name": "Vendedor Default", "reputation_score": 5.0}
            })

        # --- Transformar Reviews ---
        print("⚙️  Procesando Historial de Reseñas (Join Orders + Items)...")
        df_full = df_order_reviews.merge(df_orders, on='order_id').merge(df_order_items, on='order_id')

        lista_reviews = []
        for _, row in tqdm(df_full.head(1000).iterrows(), total=1000, desc="Reseñas"):
            lista_reviews.append({
                "order_id_pg": str(row['order_id']),
                "product_id_pg": str(row['product_id']),
                "customer": {"customer_id_pg": str(row['customer_id']), "first_name": "N/A"},
                "score": int(row['review_score']),
                "creation_date": str(row['review_creation_date'])
            })

        print(f"✅ Transformación finalizada en {round(time.time() - start_time, 2)}s.")
        return lista_productos, lista_reviews

    except Exception as e:
        print(f"❌ ERROR en Etapa 2: {e}")
        return [], []

# ETAPA 3: Carga de Datos

def load_data_only(productos, reseñas):
    print_banner("Etapa 3: Ingestión a MongoDB Atlas (Carga Base)")
    if productos: db['products'].insert_many(productos)
    if reseñas: db['reviews'].insert_many(reseñas)
    print("✅ Documentos insertados exitosamente.")


# ETAPA 4: Pruebas A/B, Optimización y Analítica
def optimizacion_y_evidencias():
    print_banner("Etapa 4: Evidencia Cuantitativa de Rendimiento (Antes vs Después)")

    pipeline_base = [
        {"$match": {"score": {"$lt": 3}}},
        {"$group": {"_id": "$product_id_pg", "promedio_score": {"$avg": "$score"}, "total_reviews": {"$sum": 1}}},
        {"$sort": {"total_reviews": -1}},
        {"$limit": 5}
    ]

    explain_cmd = {"explain": {"aggregate": "reviews", "pipeline": pipeline_base, "cursor": {}}, "verbosity": "executionStats"}

    # --- FASE 1: EL "ANTES" ---
    print("⏳ 1. Midiendo rendimiento ANTES de la optimización...")
    db['reviews'].drop_indexes()

    res_antes = db.command(explain_cmd)
    stats_antes = res_antes.get('stages', [{}])[0].get('$cursor', {}).get('executionStats', res_antes.get('executionStats', {}))
    t_antes = stats_antes.get('executionTimeMillis', 0)
    docs_antes = stats_antes.get('totalDocsExamined', 0)

    # --- FASE 2: OPTIMIZACIÓN ---
    print("⚡ 2. Creando índices compuestos y parciales...")
    start_idx = time.time()
    db['products'].create_index([("product_id_pg", 1)], unique=True)
    db['reviews'].create_index([("product_id_pg", 1), ("score", -1)])
    db['reviews'].create_index([("score", 1)], partialFilterExpression={"score": {"$lt": 3}})
    print(f"✅ Índices creados en {round(time.time() - start_idx, 2)}s.")

    # --- FASE 3: EL "DESPUÉS" ---
    print("🚀 3. Midiendo rendimiento DESPUÉS de la optimización...")
    res_despues = db.command(explain_cmd)
    stats_despues = res_despues.get('stages', [{}])[0].get('$cursor', {}).get('executionStats', res_despues.get('executionStats', {}))
    plan_despues = res_despues.get('stages', [{}])[0].get('$cursor', {}).get('queryPlanner', {}).get('winningPlan', {})

    t_despues = stats_despues.get('executionTimeMillis', 0)
    docs_despues = stats_despues.get('totalDocsExamined', 0)
    ahorro = ((docs_antes - docs_despues) / max(1, docs_antes)) * 100

    # --- REPORTE COMPARATIVO ---
    print("\n📊 CUADRO COMPARATIVO DE RENDIMIENTO")
    tabla_comparativa = [
        ["Estrategia de Búsqueda", "COLLSCAN (Escaneo Secuencial)", "IXSCAN (Uso de Índice)"],
        ["Documentos Escaneados", f"{docs_antes}", f"{docs_despues} (Ahorro del {ahorro:.2f}%)"],
        ["Tiempo de Ejecución", f"{t_antes} ms", f"{t_despues} ms"]
    ]
    print(tabulate(tabla_comparativa, headers=["Métrica", "ANTES (Sin Índices)", "DESPUÉS (Con Índices)"], tablefmt="fancy_grid"))

    # --- MÓDULO ANALÍTICO FINAL CON LOOKUP ---
    print("\n" + "="*80)
    print("📈 RESULTADO FINAL DEL MÓDULO ANALÍTICO (TOP 5)")
    print("="*80)

    pipeline_completo = pipeline_base + [
        {"$lookup": {"from": "products", "localField": "_id", "foreignField": "product_id_pg", "as": "info"}},
        {"$unwind": {"path": "$info", "preserveNullAndEmptyArrays": True}},
        {"$project": {"_id": 1, "product_name": {"$ifNull": ["$info.name", "Nombre no disponible"]}, "promedio_score": 1, "total_reviews": 1}}
    ]

    resultados = list(db['reviews'].aggregate(pipeline_completo))
    for res in resultados:
        print(f"Producto: {res['_id']} - {res['product_name'][:25]} | Promedio: {round(res['promedio_score'], 1)} | Reseñas Críticas: {res['total_reviews']}")

    print("\n" + "="*80)
    print("🔎 DETALLES TÉCNICOS DEL PLAN DE EJECUCIÓN GANADOR")
    print("="*80)
    print(json_util.dumps(plan_despues, indent=2))

# EJECUCIÓN
if __name__ == "__main__":
    setup_collections()
    data_productos, data_reviews = get_prepared_data()

    if data_productos or data_reviews:
        load_data_only(data_productos, data_reviews)
        optimizacion_y_evidencias()

    print_banner("PIPELINE DEL MÓDULO ANALÍTICO FINALIZADO EXITOSAMENTE")


🚀 ETAPA 1: PREPARACIÓN DE INFRAESTRUCTURA Y ESQUEMAS
✅ Esquemas JSON aplicados correctamente en 0.18s.

🚀 ETAPA 2: EXTRACCIÓN Y TRANSFORMACIÓN DE DATOS
Mounted at /content/drive
📥 Cargando datasets desde Google Drive...
⚙️  Procesando Catálogo de Productos...


Productos:   0%|          | 0/1000 [00:00<?, ?it/s]

⚙️  Procesando Historial de Reseñas (Join Orders + Items)...


Reseñas:   0%|          | 0/1000 [00:00<?, ?it/s]

✅ Transformación finalizada en 4.15s.

🚀 ETAPA 3: INGESTIÓN A MONGODB ATLAS (CARGA BASE)
✅ Documentos insertados exitosamente.

🚀 ETAPA 4: EVIDENCIA CUANTITATIVA DE RENDIMIENTO (ANTES VS DESPUÉS)
⏳ 1. Midiendo rendimiento ANTES de la optimización...
⚡ 2. Creando índices compuestos y parciales...
✅ Índices creados en 0.22s.
🚀 3. Midiendo rendimiento DESPUÉS de la optimización...

📊 CUADRO COMPARATIVO DE RENDIMIENTO
╒════════════════════════╤═══════════════════════════════╤═════════════════════════╕
│ Métrica                │ ANTES (Sin Índices)           │ DESPUÉS (Con Índices)   │
╞════════════════════════╪═══════════════════════════════╪═════════════════════════╡
│ Estrategia de Búsqueda │ COLLSCAN (Escaneo Secuencial) │ IXSCAN (Uso de Índice)  │
├────────────────────────┼───────────────────────────────┼─────────────────────────┤
│ Documentos Escaneados  │ 1000                          │ 186 (Ahorro del 81.40%) │
├────────────────────────┼───────────────────────────────┼──────────────